<!-- GENERATED by scripts/gen_notebooks.py. Do not edit this cell. -->
# 6 · Pretraining and the Hugging Face stack

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/nlp-llms/blob/main/notebooks/06-pretraining-huggingface.ipynb)

**From Traditional NLP to Modern LLMs** · Day 2 · 120 minutes (55 briefing, 55 lab, 10 debrief)

Subword tokenization, masked and causal pretraining, and using pretrained models through Hugging Face.

**By the end of this notebook you can:**

- Explain subword tokenization and the masked and causal pretraining objectives
- Load, inspect and run pretrained models with Hugging Face
- Configure a supplied fine-tuning run for an encoder classifier and evaluate it against Labs 1 and 2

**Stack:** Hugging Face, PyTorch

In [ ]:
#@title Workshop harness: run this cell first { display-mode: "form" }
# Generated by scripts/gen_notebooks.py: do not edit this cell.
# Tick WORKED_EXAMPLE to run every reference solution in place of your code. That gives a
# complete worked example; it does not show that you did the exercises.
WORKED_EXAMPLE = False  #@param {type:"boolean"}

import os as _os
import time as _time

from IPython import get_ipython as _get_ipython

_NOTEBOOK = "06-pretraining-huggingface"
_CONTENT_SHA = "b2b9ca33b721af21"
_EXERCISES = {1: ('pair_counts', 'merge_pair'), 2: ('tokens_per_word',), 3: ('lm_perplexity',), 4: ('mask_tokens',), 5: ('tokenize', 'compute_metrics'), 6: ('attention_entropy',)}
_RECORDED_SETTINGS = ('NLP_LLMS_OFFLINE_TINY', 'NLP_LLMS_STUB', 'NLP_LLMS_LAB07_OFFLINE', 'NLP_LLMS_LAB09_OFFLINE', 'NLP_LLMS_LAB10_OFFLINE', 'NLP_LLMS_LAB13_OFFLINE', 'NLP_LLMS_LAB14_OFFLINE', 'NLP_LLMS_LAB15_OFFLINE', 'NLP_LLMS_QUICK', 'HF_HUB_OFFLINE')
_FALLBACK_MARKERS = ('Could not load ', 'did not load', 'USING StubProvider', 'LSA stand-in fitted', 'OFFLINE TEST MODE')
_RECORD_PACKAGES = (
    "torch", "transformers", "numpy", "scikit-learn", "peft", "datasets", "openai",
    "anthropic", "typesafe-sdk", "langgraph", "langchain-core", "llama-index-core",
)


class _Workshop:
    """Keeps the folded solutions from replacing your code, and says whose code each
    checkpoint checked: yours, the reference solution, or code the lab provides."""

    _MISSING = object()

    def __init__(self, old=None):
        # Rerunning this cell (as Run all does) keeps the stored solutions but starts a new
        # run: the checkpoint results, cell times and run clock describe this run only.
        self.ref = getattr(old, "ref", {})
        self.stub = getattr(old, "stub", {})
        self.chosen = getattr(old, "chosen", set())  # exercises switched by use_reference
        self.verified = {}
        self._verifying = None
        self.results = {}
        self.cell_seconds = []
        self.cell_errors = 0
        self.fallbacks = set()
        self.started = _time.time()
        self._current = None
        self._t0 = None

    @property
    def worked(self):
        if _os.environ.get("NLP_LLMS_WORKED") == "1":
            return True
        return bool(_get_ipython().user_ns.get("WORKED_EXAMPLE"))

    def _is_reference(self, n, name, obj):
        return any(obj is r for r in self.ref.get(n, {}).get(name, ()))

    def solution(self, n):
        """Decorator on a reference solution: stores it and, unless WORKED_EXAMPLE is
        ticked, leaves your own definition of the same name in place."""

        def bind(obj, name=None):
            name = name or obj.__name__
            self.ref.setdefault(n, {}).setdefault(name, []).append(obj)
            current = _get_ipython().user_ns.get(name, self._MISSING)
            if current is not self._MISSING and not self._is_reference(n, name, current):
                self.stub.setdefault(n, {})[name] = current
            if self.worked:
                return obj
            if current is not self._MISSING and not self._is_reference(n, name, current):
                self.chosen.discard(n)  # your TODO cell ran again: back to your code
                return current
            if n in self.chosen:
                return obj  # you chose the reference with use_reference(n)
            # Your TODO cell has not run, or the name still holds a reference from a
            # worked run: go back to your code if it is known, never silently to the reference.
            yours = self.stub.get(n, {}).get(name, self._MISSING)
            if yours is not self._MISSING:
                return yours
            if callable(obj):
                return self._not_run(n, name)
            print(f"[workshop] Run your # TODO {n} cell: {name} is the reference until you do.")
            return obj

        return bind

    def _not_run(self, n, name):
        """Stands in for a function or class whose TODO cell has not run yet."""

        def missing(*args, **kwargs):
            raise NotImplementedError(
                f"{name}: run your # TODO {n} cell first, or go on with"
                f" workshop.use_reference({n!r})"
            )

        missing.__name__ = name
        return missing

    def solution_value(self, n, name, value):
        """The same as solution(n), for a value rather than a function or class."""
        if value is None or isinstance(value, (bool, int, float, str, bytes)):
            # Python shares equal small values, so yours and the reference could not be told
            # apart: give such a value in the stub instead of making it part of the exercise.
            kind = type(value).__name__
            raise TypeError(f"solution_value({n!r}, {name!r}): use an object, not a plain {kind}")
        return self.solution(n)(value, name)

    def use_reference(self, n):
        """Go on with the reference solution of exercise n in place of yours."""
        if n not in self.ref:
            raise KeyError(f"Exercise {n} has no reference solution yet: run its Solution cell")
        ns = _get_ipython().user_ns
        for name, objs in self.ref[n].items():
            ns[name] = objs[-1]
        self.chosen.add(n)
        print(
            f"Exercise {n}: now using the REFERENCE solution. Rerun the cells below it."
            f" To switch back to your code, rerun your # TODO {n} cell."
        )

    def checkpoint(self, n=None, label=None):
        """First line of a checkpoint cell: notes whose code the cell is about to check."""
        names = _EXERCISES.get(n, ())
        ns = _get_ipython().user_ns
        k = sum(self._is_reference(n, x, ns.get(x, self._MISSING)) for x in names)
        if not names:
            whose = None  # the results of the cells above: provided code, maybe your earlier work
        elif k == len(names):
            whose = "the REFERENCE solution"
        elif k == 0:
            whose = "your code"
        else:
            whose = "partly the REFERENCE solution"
        self._current = (n, str(label if label is not None else n), whose)

    def _pre(self, info):
        self._t0 = _time.time()

    def _post(self, result):
        if self._t0 is None:  # this harness cell itself: its hooks were registered mid-cell
            return
        self.cell_seconds.append(round(_time.time() - self._t0, 2))
        if not result.success and self._verifying is None:
            self.cell_errors += 1  # a verification copy fails on purpose; it is not an error
        if self._current is None:
            return
        n, label, whose = self._current
        self._current = None
        if self._verifying is not None:
            self.verified[self._verifying] = bool(result.success)
            return
        self.results[label] = (bool(result.success), whose)
        if whose is None:
            if result.success:
                print(f"[workshop] Checkpoint {label} passed.")
            elif isinstance(result.error_in_exec, NotImplementedError):
                print(
                    f"[workshop] Checkpoint {label} uses an exercise you have not written yet."
                    " Finish it, or go on with workshop.use_reference(N) for that exercise."
                )
            else:
                print(f"[workshop] Checkpoint {label} failed. Read the message above.")
        elif result.success:
            print(f"[workshop] Checkpoint {label} passed on {whose}.")
        elif isinstance(result.error_in_exec, NotImplementedError):
            print(
                f"[workshop] Checkpoint {label}: TODO {n} is not written yet. Write it, run"
                " its cell, then rerun this one. To go on without it:"
                f" workshop.use_reference({n!r})"
            )
        elif whose == "the REFERENCE solution":
            print(
                f"[workshop] Checkpoint {label} failed on the reference solution: a problem"
                " with the lab or the runtime, not with your code. Tell the instructor."
            )
        else:
            print(
                f"[workshop] Checkpoint {label} failed on {whose}. Fix TODO {n}, or go on"
                f" with workshop.use_reference({n!r})."
            )

    def _verify_begin(self, n):
        """Test-only (scripts/test_notebooks.py --verify-checkpoints): bind exercise n's
        stubs, so that a copy of its checkpoint runs on unfinished code."""
        ns = _get_ipython().user_ns
        self._saved = {name: ns.get(name, self._MISSING) for name in _EXERCISES.get(n, ())}
        ns.update(self.stub.get(n, {}))
        self._verifying = n

    def _verify_end(self, n):
        """Test-only: restore the reference and require that the stub copy failed."""
        ns = _get_ipython().user_ns
        for name, value in self._saved.items():
            if value is self._MISSING:
                ns.pop(name, None)
            else:
                ns[name] = value
        self._verifying = None
        if self.verified.get(n) is not False:
            raise AssertionError(
                f"The checkpoint of exercise {n} passed on the unfinished stub: it cannot tell"
                " a finished exercise from an unfinished one"
            )

    def summary(self):
        """Which checkpoints passed, and on whose code."""
        groups = {"your code": [], "the reference": [], "other checks": [], "failed": []}
        for label, (ok, whose) in self.results.items():
            if not ok:
                groups["failed"].append(label)
            elif whose is None:
                groups["other checks"].append(label)
            elif whose == "your code":
                groups["your code"].append(label)
            else:
                groups["the reference"].append(label)
        mode = "WORKED EXAMPLE: reference solutions" if self.worked else "your code"
        print(f"{_NOTEBOOK}, run with {mode}.")
        lines = {
            "your code": "Checkpoints passed on your code",
            "the reference": "Checkpoints passed on a reference solution",
            "other checks": "Other checks passed",
            "failed": "Checkpoints failed",
        }
        for group, labels in groups.items():
            print(f"  {lines[group]}: {', '.join(labels) or 'none'}")
        if self.worked:
            print("  A worked-example run shows how the lab goes, not that you did it.")

    def run_record(self):
        """A run record for runs/ (see runs/README.md): paste it into
        scripts/add_run_record.py. It holds no code, outputs, keys or names."""
        import importlib.metadata as _md
        import json
        import platform
        import sys

        gpu = None
        try:
            import torch

            if torch.cuda.is_available():
                gpu = torch.cuda.get_device_name(0)
        except Exception:
            pass
        packages = {}
        for name in _RECORD_PACKAGES:
            try:
                packages[name] = _md.version(name)
            except Exception:
                pass
        import datetime as _dt

        record = {
            "date": _dt.datetime.now(_dt.timezone.utc).date().isoformat(),
            "notebook": _NOTEBOOK,
            "content_sha": _CONTENT_SHA,
            "mode": "worked" if self.worked else "learner",
            # A run passes when it reached this last cell with no error on the way and every
            # checkpoint passed. Run all reruns the harness first, so it starts clean.
            "status": "pass"
            if self.results and self.cell_errors == 0 and all(ok for ok, _ in self.results.values())
            else "fail",
            "cell_errors": self.cell_errors,
            "fallbacks": sorted(self.fallbacks),
            "seconds": round(_time.time() - self.started, 1),
            "cell_seconds": self.cell_seconds,
            "python": platform.python_version(),
            "platform": sys.platform,
            "gpu": gpu,
            "colab_release": _os.environ.get("COLAB_RELEASE_TAG"),
            "provider": _get_ipython().user_ns.get("PROVIDER"),
            "jev_path": _get_ipython().user_ns.get("JEV_PATH"),
            "settings": {k: _os.environ[k] for k in _RECORDED_SETTINGS if _os.environ.get(k)},
            "packages": packages,
        }
        print("----- run record (paste into scripts/add_run_record.py) -----")
        print(json.dumps(record))
        print("----- end of run record -----")


_ip = _get_ipython()
_old = _ip.user_ns.get("workshop")
# Remove only an earlier harness's own hooks; every other hook (Colab's, an extension's)
# stays registered.
for _event in ("pre_run_cell", "post_run_cell"):
    for _callback in list(_ip.events.callbacks[_event]):
        if type(getattr(_callback, "__self__", None)).__name__ == "_Workshop":
            _ip.events.unregister(_event, _callback)
workshop = _Workshop(_old if hasattr(_old, "ref") else None)

# Notice when a lab falls back to a stand-in, for the run record. print is wrapped once.
import builtins as _builtins

if not getattr(_builtins.print, "_workshop", False):
    _print = _builtins.print

    def _watching_print(*args, **kwargs):
        try:
            ip = _get_ipython()
            ws = ip.user_ns.get("workshop") if ip is not None else None
            if ws is not None and hasattr(ws, "fallbacks"):
                # Labs print their fallback notices as strings; never stringify other objects.
                text = " ".join(a for a in args if isinstance(a, str))
                ws.fallbacks.update(m.strip() for m in _FALLBACK_MARKERS if m in text)
        except Exception:
            pass  # watching must never break printing
        return _print(*args, **kwargs)

    _watching_print._workshop = True
    _builtins.print = _watching_print
_ip.events.register("pre_run_cell", workshop._pre)
_ip.events.register("post_run_cell", workshop._post)
print("Workshop harness ready:", "WORKED EXAMPLE" if workshop.worked else "checking your code")


## Setup (3 minutes)

**Use a GPU runtime:** *Runtime → Change runtime type → T4 GPU*. On a GPU the lab fine-tunes DistilBERT on the full training split. On a CPU runtime it still runs, but Exercise 5 fine-tunes a much smaller encoder on a subset of the training data, and the notebook says so beside every number.

No API key is needed. The lab downloads three pretrained checkpoints from the Hugging Face Hub (several hundred megabytes in total). Downloads without a Hugging Face token are allowed; the library may print a warning about a missing `HF_TOKEN` or about rate limits, which you can ignore.

**How the lab works.** There are five exercises and one optional stretch section. Each exercise has:

1. a **Predict** question to answer before you run anything;
2. a `# TODO` cell where you write one or two short functions;
3. a folded **Solution** cell (open it only after trying);
4. a **Checkpoint** cell that tells you whether your function is right.

The folded solution cells never replace your code: each checkpoint checks your own functions and says so. If you are stuck, `workshop.use_reference(N)` lets you go on with exercise N's reference solution, and the checkpoint then says it checked the reference. To run the whole lab as a worked example, tick `WORKED_EXAMPLE` in the harness cell at the top; a worked run shows how the lab goes, not that you did it.

In [ ]:
# Setup: pinned installs.
# Colab preinstalls PyTorch, NumPy, scikit-learn and matplotlib. The Hugging Face libraries
# are pinned to the versions this notebook was written against; any that already match are skipped.
import importlib.metadata
import subprocess
import sys

PINS = {"transformers": "5.18.0", "tokenizers": "0.23.2", "datasets": "5.0.1",
        "accelerate": "1.15.0"}


def installed_version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None


_to_install = [f"{name}=={v}" for name, v in PINS.items() if installed_version(name) != v]
if _to_install:
    print("Installing", " ".join(_to_install))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_to_install], check=True)

In [ ]:
# Imports, seeds and device.
import json
import math
import os
import random
import re
import threading
import time
from collections import Counter
from pathlib import Path

import datasets
import matplotlib.pyplot as plt
import numpy as np
import tokenizers
import torch
import torch.nn.functional as F
import transformers
from datasets import Dataset
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer
from transformers import (
    AutoModel,
    AutoModelForCausalLM,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

for _name, _version in PINS.items():
    assert installed_version(_name) == _version, (
        f"{_name} {installed_version(_name)} is loaded but {_version} is pinned: "
        "use Runtime → Restart session, then run all again"
    )

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
transformers.set_seed(SEED)
datasets.disable_progress_bars()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NAVY, ACCENT = "#16324f", "#b3541e"  # plot colors
LAB_START = time.time()
print(f"PyTorch {torch.__version__} · transformers {transformers.__version__} · "
      f"tokenizers {tokenizers.__version__} · datasets {datasets.__version__}")
print("Device:", DEVICE, f"({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else
      "(no GPU: Exercise 5 will run the smaller CPU configuration)")

## Data

The classification set of the running thread: arXiv Topics v1, the titles and abstracts of 7,000 arXiv papers in four classes (Language, Vision, Machine learning, Robotics). Lab 1 set the baseline on it with TF-IDF and logistic regression, and Lab 2 with averaged word embeddings. This lab adds a fine-tuned pretrained encoder, on the **same** 1,600 test papers, with the same two metrics.

The next cell is the workshop's standard loading cell, the same in every lab (see `data/README.md` in the repository). It checks the file against a hash, so everyone gets the same bytes and the same split. The rule: fit on **train**, choose settings on **val**, report on **test** once per model.

In [ ]:
import csv, gzip, hashlib, io, os, urllib.request
from pathlib import Path


def fetch(name, urls, sha256):
    """Return the bytes of a workshop data file, checked against its SHA-256."""
    cache = Path(os.environ.get("NLP_LLMS_DATA", "data"))
    for path in (cache / name, Path("../data") / name):
        if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == sha256:
            return path.read_bytes()
    for url in urls:  # canonical URL first, fallback second
        try:
            with urllib.request.urlopen(url, timeout=60) as response:
                blob = response.read()
        except OSError as error:
            print(f"Could not fetch {url}: {error}")
            continue
        if hashlib.sha256(blob).hexdigest() != sha256:
            print(f"Ignoring {url}: the file does not match the expected hash")
            continue
        cache.mkdir(parents=True, exist_ok=True)
        (cache / name).write_bytes(blob)
        return blob
    raise RuntimeError(f"Could not load {name} from any source")


TOPIC_CLASSES = ["cs.CL", "cs.CV", "cs.LG", "cs.RO"]  # label 0, 1, 2, 3
TOPIC_NAMES = ["Language", "Vision", "Machine learning", "Robotics"]


def load_topics():
    """arXiv Topics v1. Returns {"train" | "val" | "test": (texts, labels)}."""
    blob = fetch(
        "arxiv_topics_v1.csv.gz",
        [
            "https://raw.githubusercontent.com/project-delphi/nlp-llms/main/data/arxiv_topics_v1.csv.gz",
            "https://cdn.jsdelivr.net/gh/project-delphi/nlp-llms@main/data/arxiv_topics_v1.csv.gz",
        ],
        "49b51502fc6cf6ecff23fa266c5372594a88414dd96513579237792710246dc5",
    )
    splits = {"train": ([], []), "val": ([], []), "test": ([], [])}
    for row in csv.DictReader(io.StringIO(gzip.decompress(blob).decode("utf-8"))):
        texts, labels = splits[row["split"]]
        texts.append(row["title"] + "\n" + row["abstract"])
        labels.append(int(row["label"]))
    return splits


def load_lm_corpus():
    """Tiny Shakespeare. Returns {"train" | "val" | "test": str}, split by character offset."""
    blob = fetch(
        "tinyshakespeare.txt",
        [
            "https://raw.githubusercontent.com/karpathy/char-rnn/6f9487a6fe5b420b7ca9afb0d7c078e37c1d1b4e/data/tinyshakespeare/input.txt",
            "https://raw.githubusercontent.com/project-delphi/nlp-llms/main/data/tinyshakespeare.txt",
        ],
        "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed",
    )
    text = blob.decode("utf-8")
    return {"train": text[:1_000_000], "val": text[1_000_000:1_055_000], "test": text[1_055_000:]}

In [ ]:
topics = load_topics()  # {"train" | "val" | "test": (texts, labels)}
train_texts, train_labels = topics["train"]
val_texts, val_labels = topics["val"]
test_texts, test_labels = topics["test"]

for split, (texts, labels) in topics.items():
    print(f"{split:<5} {len(texts):>5,} papers, per class {np.bincount(labels).tolist()}")
print(f"\nDevice: {DEVICE}")

## The pretrained checkpoints

A **checkpoint** is a saved set of parameters together with its configuration and tokenizer files, stored on the Hugging Face Hub under an ID. This lab uses three. Read a model card before you use a model: it states the training data, intended use, limits and license.

| Role | Checkpoint | Size | Used in |
|---|---|---|---|
| Masked LM (encoder) | [`distilbert/distilbert-base-uncased`](https://huggingface.co/distilbert/distilbert-base-uncased) | 66M parameters, 6 layers, $d = 768$, WordPiece vocabulary of 30,522 | Exercises 2 and 4, the stretch, and Exercise 5 on a GPU |
| Small encoder | [`google/bert_uncased_L-4_H-256_A-4`](https://huggingface.co/google/bert_uncased_L-4_H-256_A-4) (BERT-mini) | 11M parameters, 4 layers, $d = 256$, the same vocabulary | Exercise 5 on a CPU runtime |
| Causal LM (decoder) | [`distilbert/distilgpt2`](https://huggingface.co/distilbert/distilgpt2) | 82M parameters, byte-level BPE vocabulary of 50,257 | Exercises 2 and 3 |

All three are released under the Apache 2.0 license. The next cells start the downloads in the background, so they run while you work on Exercise 1, which needs no model.

In [ ]:
# Checkpoint IDs. The workshop records them in _variables.yml; Colab cannot read that file,
# so they are repeated here.
MLM_ENCODER = "distilbert/distilbert-base-uncased"  # masked LM; fine-tuned on a GPU
CPU_ENCODER = "google/bert_uncased_L-4_H-256_A-4"  # BERT-mini; fine-tuned on a CPU runtime
CAUSAL_LM = "distilbert/distilgpt2"  # causal LM
FINETUNE_ENCODER = MLM_ENCODER if DEVICE == "cuda" else CPU_ENCODER

# TEST-ONLY SWITCH. Participants: leave this alone; it is off unless the environment sets it.
# The repository's test machine cannot reach the Hugging Face Hub. With NLP_LLMS_OFFLINE_TINY=1
# the next (folded) cell replaces the three checkpoints with tiny, randomly initialized models of
# the same families, so every cell's code can be executed offline. Numbers printed in that mode
# are meaningless and must never be reported.
OFFLINE_TINY = os.environ.get("NLP_LLMS_OFFLINE_TINY") == "1"
print("Fine-tuning encoder for this runtime:", FINETUNE_ENCODER)

In [ ]:
#@title Test-only: offline stand-ins (does nothing on Colab) { display-mode: "form" }
if OFFLINE_TINY:
    from tokenizers import decoders, normalizers, pre_tokenizers, processors
    from tokenizers.models import WordPiece
    from tokenizers.trainers import WordPieceTrainer
    from transformers import (BertConfig, BertForMaskedLM, DistilBertConfig, DistilBertForMaskedLM,
                              GPT2Config, GPT2LMHeadModel, PreTrainedTokenizerFast)

    _tiny = Path("offline-tiny")
    # A lowercasing WordPiece tokenizer with BERT's special tokens, trained on the training texts.
    _wp = Tokenizer(WordPiece(unk_token="[UNK]"))
    _wp.normalizer = normalizers.BertNormalizer(lowercase=True)
    _wp.pre_tokenizer = pre_tokenizers.BertPreTokenizer()
    _wp.decoder = decoders.WordPiece()
    _wp.train_from_iterator(train_texts, WordPieceTrainer(
        vocab_size=8000, special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]))
    _wp.post_processor = processors.TemplateProcessing(
        single="[CLS] $A [SEP]", pair="[CLS] $A [SEP] $B [SEP]",
        special_tokens=[("[CLS]", _wp.token_to_id("[CLS]")), ("[SEP]", _wp.token_to_id("[SEP]"))])
    _enc_tok = PreTrainedTokenizerFast(
        tokenizer_object=_wp, unk_token="[UNK]", pad_token="[PAD]", cls_token="[CLS]",
        sep_token="[SEP]", mask_token="[MASK]", model_max_length=512)
    # A byte-level BPE tokenizer with GPT-2's end-of-text token.
    _bl = Tokenizer(BPE())
    _bl.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    _bl.decoder = decoders.ByteLevel()
    _bl.train_from_iterator(train_texts, BpeTrainer(
        vocab_size=8000, special_tokens=["<|endoftext|>"],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
    _clm_tok = PreTrainedTokenizerFast(
        tokenizer_object=_bl, bos_token="<|endoftext|>", eos_token="<|endoftext|>",
        unk_token="<|endoftext|>", model_max_length=1024)
    # Tiny random models of the three families, saved like Hub checkpoints.
    _V = len(_enc_tok)
    _stand_ins = {
        "distilbert": (DistilBertForMaskedLM(DistilBertConfig(
            vocab_size=_V, dim=64, n_layers=2, n_heads=2, hidden_dim=128,
            pad_token_id=_enc_tok.pad_token_id)), _enc_tok),
        "bert-mini": (BertForMaskedLM(BertConfig(
            vocab_size=_V, hidden_size=64, num_hidden_layers=2, num_attention_heads=2,
            intermediate_size=128, pad_token_id=_enc_tok.pad_token_id)), _enc_tok),
        "distilgpt2": (GPT2LMHeadModel(GPT2Config(
            vocab_size=len(_clm_tok), n_embd=64, n_layer=2, n_head=2, n_positions=1024,
            bos_token_id=_clm_tok.bos_token_id, eos_token_id=_clm_tok.eos_token_id)), _clm_tok),
    }
    for _name, (_model, _tok) in _stand_ins.items():
        _model.save_pretrained(_tiny / _name)
        _tok.save_pretrained(_tiny / _name)
    MLM_ENCODER = str(_tiny / "distilbert")
    CPU_ENCODER = str(_tiny / "bert-mini")
    CAUSAL_LM = str(_tiny / "distilgpt2")
    FINETUNE_ENCODER = MLM_ENCODER if DEVICE == "cuda" else CPU_ENCODER
    print("=" * 72)
    print("OFFLINE TEST MODE: tiny random stand-in models. Every number below is meaningless.")
    print("=" * 72)

In [ ]:
# Start downloading the checkpoints in a background thread; Exercise 1 does not need them.
from huggingface_hub import hf_hub_download, snapshot_download

_download_errors = []
# Configuration, tokenizer files and PyTorch weights only; the repositories also hold weights
# for other frameworks that this lab does not need.
_FILES = ["config.json", "generation_config.json", "tokenizer.json", "tokenizer_config.json",
          "special_tokens_map.json", "vocab.txt", "vocab.json", "merges.txt", "model.safetensors"]


def _download_all():
    try:
        for repo in dict.fromkeys([MLM_ENCODER, CAUSAL_LM, FINETUNE_ENCODER]):
            local = snapshot_download(repo, allow_patterns=_FILES)
            if not (Path(local) / "model.safetensors").exists():  # an older repository
                hf_hub_download(repo, "pytorch_model.bin")
    except Exception as error:  # reported by wait_for_downloads()
        _download_errors.append(error)


def wait_for_downloads():
    """Block until the background downloads have finished; re-raise any error they hit."""
    started = time.time()
    downloads.join()
    if _download_errors:
        raise _download_errors[0]
    print(f"Checkpoints ready (waited {time.time() - started:.0f} s)")


downloads = threading.Thread(target=None if OFFLINE_TINY else _download_all, daemon=True)
downloads.start()
print("Downloads started in the background.")

# Part A · Subword tokenization

## Exercise 1 · Byte-pair encoding by hand (10 minutes)

**Byte-pair encoding (BPE)** builds a vocabulary between characters and words. Starting from characters, it repeatedly glues together the most frequent adjacent pair of symbols (briefing, Section 2):

1. **Pre-tokenize:** split the corpus into words and count them.
2. **Initialize:** write each word as its characters followed by the end-of-word symbol `_`.
3. **Repeat $M$ times:** (a) count every adjacent pair, weighted by word counts; (b) pick the most frequent pair, ties going to the pair that sorts first; (c) merge it into one symbol everywhere, scanning left to right; (d) record the merge.

The algorithm works on a dictionary from words, written as tuples of symbols, to counts:

```text
{("l", "o", "w", "_"): 5, ("l", "o", "w", "e", "r", "_"): 2, ...}
```

**Predict.** On the corpus `low` ×5, `lower` ×2, `newest` ×6, `widest` ×3, which pair is merged first, and what is its count? After five merges, is `lowest` (never seen) one token, two tokens or more?

**Task.** Write steps 3a and 3c.

- `pair_counts(words)`: return a `Counter` from each adjacent pair `(a, b)` to its count, adding the word's count once per occurrence.
- `merge_pair(words, pair)`: return a new dictionary in which every occurrence of `a` immediately followed by `b` is replaced by the single symbol `a + b`, scanning each word left to right.

In [ ]:
# TODO 1: complete both functions.
def pair_counts(words):
    """words: {tuple of symbols: count}. Return a Counter {(a, b): count} of adjacent pairs."""
    counts = Counter()
    ...
    return counts


def merge_pair(words, pair):
    """Return a new {tuple of symbols: count} with each adjacent (a, b) replaced by a + b."""
    raise NotImplementedError("TODO 1")

In [ ]:
#@title Solution 1 — try it yourself first { display-mode: "form" }
@workshop.solution(1)
def pair_counts(words):
    """words: {tuple of symbols: count}. Return a Counter {(a, b): count} of adjacent pairs."""
    counts = Counter()
    for symbols, freq in words.items():
        for a, b in zip(symbols[:-1], symbols[1:], strict=True):  # each adjacent pair
            counts[a, b] += freq
    return counts


@workshop.solution(1)
def merge_pair(words, pair):
    """Return a new {tuple of symbols: count} with each adjacent (a, b) replaced by a + b."""
    a, b = pair
    merged = {}
    for symbols, freq in words.items():
        out, i = [], 0
        while i < len(symbols):  # left to right, so "a a a" with pair (a, a) becomes "aa a"
            if i + 1 < len(symbols) and symbols[i] == a and symbols[i + 1] == b:
                out.append(a + b)
                i += 2
            else:
                out.append(symbols[i])
                i += 1
        merged[tuple(out)] = merged.get(tuple(out), 0) + freq
    return merged

<details>
<summary>Why this solution works</summary>

`pair_counts` is step 3a: a pair inside a word that occurs $f$ times is seen $f$ times in the corpus, so it adds $f$, once for each place it occurs in the word. Pairs never cross a word boundary because each word is its own tuple.

`merge_pair` is step 3c. Scanning left to right and jumping two positions after a merge makes overlapping occurrences unambiguous: in `a a a` the first two symbols merge and the third stays. Each merge adds exactly one new symbol, `a + b`, which is why the vocabulary grows by one per merge (@eq-bpe-size in the briefing).

</details>

**Provided.** The training loop from the briefing and the encoder for new words, built on your two functions.

In [ ]:
def train_bpe(word_freqs, num_merges):
    """Learn an ordered list of BPE merges from {word: count}."""
    words = {tuple(w) + ("_",): f for w, f in word_freqs.items()}  # {("l", "o", "w", "_"): 5, ...}
    merges = []
    for _ in range(num_merges):
        counts = pair_counts(words)  # step 3a
        if not counts:
            break  # every word is one symbol
        best = min(counts, key=lambda p: (-counts[p], p))  # step 3b: most frequent, ties sort first
        words = merge_pair(words, best)  # step 3c
        merges.append(best)  # step 3d
    return merges


def bpe_encode(word, merges):
    """Split a word into characters plus "_", then apply the merges in the learned order."""
    symbols = tuple(word) + ("_",)
    for pair in merges:
        (symbols,) = merge_pair({symbols: 1}, pair)
    return list(symbols)

In [ ]:
# Checkpoint 1
workshop.checkpoint(1)
TOY = {"low": 5, "lower": 2, "newest": 6, "widest": 3}
_words = {tuple(w) + ("_",): f for w, f in TOY.items()}
_counts = pair_counts(_words)
assert isinstance(_counts, Counter), "pair_counts must return a Counter"
assert _counts["e", "s"] == 9, "('e', 's') is in newest (6) and widest (3): 9"
assert _counts["l", "o"] == 7 and _counts["w", "_"] == 5, "weight each pair by the word's count"
assert merge_pair({("a", "a", "a"): 1}, ("a", "a")) == {("aa", "a"): 1}, "merge left to right"
assert merge_pair({("a", "b", "a", "b"): 2}, ("a", "b")) == {("ab", "ab"): 2}, "merge all of them"

merges = train_bpe(TOY, 5)
expected = [("e", "s"), ("es", "t"), ("est", "_"), ("l", "o"), ("lo", "w")]
assert merges == expected, f"first five merges: {merges}"
assert bpe_encode("lowest", merges) == ["low", "est_"], bpe_encode("lowest", merges)
assert bpe_encode("newer", merges) == ["n", "e", "w", "e", "r", "_"], bpe_encode("newer", merges)

base = {s for word in _words for s in word}  # V_0: the 11 symbols d e i l n o r s t w _
vocab = base | {a + b for a, b in merges}
assert len(base) == 11 and len(vocab) == 11 + 5, "|V| = |V_0| + M (@eq-bpe-size)"
print("Checkpoint 1 passed")
for step, (a, b) in enumerate(merges, 1):
    print(f"  merge {step}: {a} + {b}")
print("lowest ->", bpe_encode("lowest", merges), "· newer ->", bpe_encode("newer", merges))

**Explain.** Compare with your prediction. The pair `e s` wins merge 1 with count 6 + 3 = 9; `s t` and `t _` also have count 9 and lose the tie. After five merges the algorithm has found a suffix, `est_`, and a stem, `low`, without being told what either is, and the unseen word `lowest` costs two tokens. `newer` uses only rare pieces, so it stays at the character level: frequent strings are cheap, rare ones cost several tokens.

## Exercise 2 · Train a tokenizer and count tokens per word (7 minutes)

The `tokenizers` library runs the same algorithm at full speed. The next cell after the checkpoint trains a BPE tokenizer with a 4,000-entry vocabulary on the 4,800 training texts, then compares it with the two pretrained tokenizers: DistilBERT's WordPiece (30,522 entries, lowercased) and GPT-2's byte-level BPE (50,257 entries).

The comparison metric is **tokens per word**: the total number of tokens divided by the total number of whitespace-separated words, over the validation texts, special tokens excluded. Sequence length, context limits and cost are all counted in tokens.

**Predict.** Will our 4,000-entry vocabulary give more or fewer tokens per word than the 30,000-entry WordPiece vocabulary? By roughly how much?

**Task.** Complete `tokens_per_word(encode_fn, texts)`, where `encode_fn` maps one string to a list of tokens (or token IDs).

In [ ]:
# TODO 2: complete this function.
def tokens_per_word(encode_fn, texts):
    """Total tokens over total whitespace-separated words, summed over `texts`."""
    raise NotImplementedError("TODO 2")

In [ ]:
#@title Solution 2 — try it yourself first { display-mode: "form" }
@workshop.solution(2)
def tokens_per_word(encode_fn, texts):
    """Total tokens over total whitespace-separated words, summed over `texts`."""
    n_tokens = sum(len(encode_fn(text)) for text in texts)
    n_words = sum(len(text.split()) for text in texts)
    return n_tokens / n_words  # a ratio of totals, not a mean of per-text ratios

<details>
<summary>Why this solution works</summary>

Dividing the two totals weights every word equally. Averaging the per-text ratios instead would give a short abstract the same weight as a long one. A whitespace tokenizer produces exactly one token per word, so it scores 1.0; any subword tokenizer scores at least close to 1, and more when it splits words or punctuation.

</details>

In [ ]:
# Checkpoint 2
workshop.checkpoint(2)
_s = "We fine-tune a pretrained transformer, then evaluate it."
assert tokens_per_word(str.split, [_s]) == 1.0, "a whitespace tokenizer gives exactly 1.0"
assert tokens_per_word(list, ["ab cd", "e"]) == 6 / 3, "characters: 6 (with the space) over 3 words"
print("Checkpoint 2 passed")

**Run.** Train our tokenizer, then compare. The pretrained tokenizers are loaded by checkpoint ID with `AutoTokenizer`; a tokenizer always comes from the same checkpoint as its model.

In [ ]:
BPE_VOCAB = 4000
started = time.time()
our_bpe = Tokenizer(BPE(unk_token="[UNK]"))
our_bpe.pre_tokenizer = Whitespace()  # step 1: split into words and punctuation
trainer = BpeTrainer(vocab_size=BPE_VOCAB, special_tokens=["[UNK]", "[PAD]"])
our_bpe.train_from_iterator(train_texts, trainer)  # steps 2 and 3
print(f"Trained a BPE tokenizer on {len(train_texts):,} texts in {time.time() - started:.1f} s")
assert our_bpe.get_vocab_size() == BPE_VOCAB, our_bpe.get_vocab_size()

wait_for_downloads()
wordpiece = AutoTokenizer.from_pretrained(MLM_ENCODER)  # DistilBERT's WordPiece
gpt2_bpe = AutoTokenizer.from_pretrained(CAUSAL_LM)  # GPT-2's byte-level BPE

encoders = {
    f"ours: BPE, {our_bpe.get_vocab_size():,} entries": lambda t: our_bpe.encode(t).tokens,
    f"{MLM_ENCODER} ({len(wordpiece):,})": wordpiece.tokenize,  # no special tokens
    f"{CAUSAL_LM} ({len(gpt2_bpe):,})": gpt2_bpe.tokenize,
}
tpw = {name: tokens_per_word(fn, val_texts) for name, fn in encoders.items()}
print(f"\n{'tokenizer':<52} tokens per word (val)")
for name, value in tpw.items():
    print(f"{name:<52} {value:.3f}")
_ours = next(iter(tpw.values()))
assert 1.45 < _ours < 1.65, f"our BPE should give about 1.55 tokens per word, got {_ours:.3f}"

sentence = "We fine-tune a pretrained transformer for multilingual summarization."
print(f"\n{sentence}")
print("ours:     ", " ".join(our_bpe.encode(sentence).tokens))
print("WordPiece:", " ".join(wordpiece.tokenize(sentence)))
print("GPT-2:    ", " ".join(gpt2_bpe.tokenize(sentence)))

**Explain.** A smaller vocabulary means fewer merges, so more words are split: our 4,000-entry tokenizer needs more tokens per word than the 30,000- and 50,000-entry ones, although it was trained on arXiv text and they were not. Look at the three tokenizations of the sentence: our tokenizer marks no word boundary at all (the `Whitespace` pre-tokenizer throws the spaces away), WordPiece marks word-continuing pieces with `##`, and GPT-2 attaches the preceding space to a word, printed as `Ġ`. None of them splits along the morphology a linguist would draw.

# Part B · Two pretraining objectives

## Exercise 3 · Causal language modeling and perplexity (8 minutes)

A **causal language model** predicts each token from the tokens before it. Its loss is the average negative log-probability (briefing, @eq-clm), and its exponential is the perplexity of Module 1:

$$
\mathcal{L}_{\text{CLM}}(\theta) = -\frac{1}{T-1} \sum_{t=2}^{T} \log P_\theta(w_t \mid w_{<t}),
\qquad
\mathrm{PPL} = \exp\bigl(\mathcal{L}_{\text{CLM}}\bigr).
$$

The first token has no left context, so the average runs over tokens 2 to $T$, as the library's loss does. The model returns logits of shape `(1, T, |V|)`: row $t - 1$ holds the scores for token $t$. So the logits must be **shifted by one position** against the input IDs.

**Predict.** Take a well-formed sentence and the same words in a random order. How much higher will the perplexity of the shuffled version be: twice, ten times, more?

**Task.** Complete `lm_perplexity(text, model, tokenizer)`.

In [ ]:
clm_tokenizer = AutoTokenizer.from_pretrained(CAUSAL_LM)
clm = AutoModelForCausalLM.from_pretrained(CAUSAL_LM).to(DEVICE).eval()
print(f"{CAUSAL_LM}: {sum(p.numel() for p in clm.parameters()) / 1e6:.1f}M parameters, "
      f"vocabulary {len(clm_tokenizer):,}")

In [ ]:
# TODO 3: complete this function.
def lm_perplexity(text, model, tokenizer):
    """Perplexity of a causal LM on `text`, averaged over tokens 2 to T."""
    ids = tokenizer(text, return_tensors="pt")["input_ids"].to(model.device)  # (1, T)
    with torch.no_grad():
        logits = model(ids).logits  # (1, T, |V|); row t - 1 scores token t
    ...

In [ ]:
#@title Solution 3 — try it yourself first { display-mode: "form" }
@workshop.solution(3)
def lm_perplexity(text, model, tokenizer):
    """Perplexity of a causal LM on `text`, averaged over tokens 2 to T."""
    ids = tokenizer(text, return_tensors="pt")["input_ids"].to(model.device)  # (1, T)
    with torch.no_grad():
        logits = model(ids).logits  # (1, T, |V|); row t - 1 scores token t
    log_probs = torch.log_softmax(logits[0, :-1].float(), dim=-1)  # (T - 1, |V|): rows 1..T-1
    targets = ids[0, 1:]  # (T - 1,): tokens 2..T
    nll = -log_probs.gather(1, targets[:, None]).squeeze(1)  # (T - 1,): -log P(w_t | w_<t)
    return math.exp(nll.mean().item())

<details>
<summary>Why this solution works</summary>

Dropping the last row of the logits and the first input ID lines each prediction up with its target: row $t - 1$ against token $t$. `log_softmax` turns scores into log-probabilities over the vocabulary in a numerically stable way, `gather` picks the log-probability of the token that actually came next, and the mean of the negatives is @eq-clm. When you pass `labels=ids`, the library does the same shift internally, which is what the checkpoint compares against. This is the cross-entropy you minimized in Lab 5, on BPE tokens instead of characters.

</details>

In [ ]:
# Checkpoint 3
workshop.checkpoint(3)
SENTENCE = "The model is trained on a large corpus of text."
_words = SENTENCE.split()
random.Random(SEED).shuffle(_words)
SHUFFLED = " ".join(_words)

ppl = lm_perplexity(SENTENCE, clm, clm_tokenizer)
ppl_shuffled = lm_perplexity(SHUFFLED, clm, clm_tokenizer)
_ids = clm_tokenizer(SENTENCE, return_tensors="pt")["input_ids"].to(DEVICE)
with torch.no_grad():
    _library = math.exp(clm(_ids, labels=_ids).loss.item())  # the library shifts internally
assert math.isclose(ppl, _library, rel_tol=1e-3), f"yours {ppl:.4f}, library {_library:.4f}"
if not OFFLINE_TINY:  # a random model has no reason to prefer word order
    assert ppl < ppl_shuffled, "a pretrained LM should find the real sentence more likely"
print("Checkpoint 3 passed\n")
print(f"{ppl:>10.1f}  {SENTENCE}")
print(f"{ppl_shuffled:>10.1f}  {SHUFFLED}")

**Run.** The same model as a generator: its distribution over the next token, and a greedy continuation (always the most likely token, Module 3).

In [ ]:
prompt = "Large language models are trained to"
ids = clm_tokenizer(prompt, return_tensors="pt")["input_ids"].to(DEVICE)
with torch.no_grad():
    next_probs = clm(ids).logits[0, -1].softmax(dim=-1)  # (|V|,) distribution over the next token
top = next_probs.topk(5)
print(f"Next token after {prompt!r}:")
for p, i in zip(top.values, top.indices, strict=True):
    print(f"  {clm_tokenizer.decode(int(i))!r:>14}  {p.item():.3f}")

with torch.no_grad():
    out = clm.generate(ids, max_new_tokens=25, do_sample=False,
                       pad_token_id=clm_tokenizer.eos_token_id)
print("\nGreedy continuation:\n" + clm_tokenizer.decode(out[0]))

# Perplexities are per token, so they compare only over the same tokens. This number is per
# GPT-2 BPE token and is NOT comparable with the per-character perplexities of Labs 1, 3 and 5.
# The total negative log-probability does not depend on the unit: divide it by the characters
# scored to get nats per character (the first token is context, so its characters are excluded).
_ids = clm_tokenizer(SENTENCE)["input_ids"]
total_nll = math.log(ppl) * (len(_ids) - 1)  # T · L summed over tokens 2..T, in nats
scored_chars = len(clm_tokenizer.decode(_ids[1:]))
print(f"\n{SENTENCE!r}: {len(_ids)} tokens, perplexity {ppl:.1f} per token, "
      f"{total_nll / scored_chars:.3f} nats per character")

**Explain.** The shuffled words are the same tokens, so any difference comes from order: a pretrained causal LM has learned which sequences English allows, and shuffling multiplies the perplexity many times over. The next-token distribution is spread over several plausible words, and the greedy continuation tends to repeat itself, as in Module 3. Nats per character is how you would put this model next to the character-level models of Labs 1, 3 and 5, but only on the **same text**: this is one arXiv-style sentence, not the Tiny Shakespeare test split.

## Exercise 4 · Masked language modeling: select and corrupt (10 minutes)

An encoder sees both sides of every position, so it cannot be trained to predict the next token: it could read it. **Masked language modeling** hides some tokens and predicts them (briefing, Section 5):

1. **Select** each non-special position independently with probability $p = 0.15$. Call the selected set $\mathcal{S}$.
2. **Corrupt** each selected position independently: with probability 0.8 replace it by `[MASK]`; with probability 0.1 by a token drawn uniformly from the vocabulary; with probability 0.1 leave it unchanged.
3. **Predict** the original token at every selected position. Positions outside $\mathcal{S}$ get the target $-100$, which PyTorch's cross-entropy ignores.

`special_mask` is `True` at `[CLS]`, `[SEP]` and `[PAD]` positions; those are never selected.

**Predict.** In a batch with about 50,000 real tokens, about how many positions will be selected, and how many of those will still show their original token to the model?

**Task.** Complete `mask_tokens(input_ids, special_mask, mask_id, vocab_size, p=0.15)`. Use `torch.bernoulli`, `torch.rand` and `torch.randint` so that `torch.manual_seed` makes the result reproducible.

In [ ]:
mlm_tokenizer = AutoTokenizer.from_pretrained(MLM_ENCODER)
mlm = AutoModelForMaskedLM.from_pretrained(MLM_ENCODER).to(DEVICE).eval()
print(f"{MLM_ENCODER}: {sum(p.numel() for p in mlm.parameters()) / 1e6:.1f}M parameters · "
      f"[MASK] = {mlm_tokenizer.mask_token_id} · vocabulary {len(mlm_tokenizer):,}")

In [ ]:
# TODO 4: complete this function.
def mask_tokens(input_ids, special_mask, mask_id, vocab_size, p=0.15):
    """input_ids: (B, T) token IDs; special_mask: (B, T) bool. Returns (corrupted, labels)."""
    labels = input_ids.clone()
    corrupted = input_ids.clone()
    # 1. select each non-special position with probability p; set labels to -100 elsewhere
    ...
    # 2. corrupt the selected positions: 80% mask_id, 10% a random token, 10% unchanged
    ...
    return corrupted, labels

In [ ]:
#@title Solution 4 — try it yourself first { display-mode: "form" }
@workshop.solution(4)
def mask_tokens(input_ids, special_mask, mask_id, vocab_size, p=0.15):
    """input_ids: (B, T) token IDs; special_mask: (B, T) bool. Returns (corrupted, labels)."""
    labels = input_ids.clone()
    corrupted = input_ids.clone()
    # 1. select each non-special position with probability p; set labels to -100 elsewhere
    probs = torch.full(input_ids.shape, p)
    probs[special_mask] = 0.0  # [CLS], [SEP], [PAD] are never selected
    selected = torch.bernoulli(probs).bool()  # (B, T): the set S
    labels[~selected] = -100  # no loss outside S
    # 2. corrupt the selected positions: 80% mask_id, 10% a random token, 10% unchanged
    u = torch.rand(input_ids.shape)  # one uniform draw per position decides its branch
    to_mask = selected & (u < 0.8)
    to_random = selected & (u >= 0.8) & (u < 0.9)  # the remaining 10% (u >= 0.9) stay as they are
    corrupted[to_mask] = mask_id
    corrupted[to_random] = torch.randint(vocab_size, (int(to_random.sum()),))
    return corrupted, labels

<details>
<summary>Why this solution works</summary>

`torch.bernoulli` with probability 0.15 everywhere and 0 at special positions is step 1. One uniform number $u$ per position then picks the branch: $u < 0.8$ is `[MASK]`, $0.8 \le u < 0.9$ is a random token, $u \ge 0.9$ is unchanged, which gives 80/10/10 among the selected positions. The labels keep the *original* ID at every selected position, including the unchanged ones, because the model must predict there too: it cannot tell which visible tokens are real. That is why `[MASK]` alone would not be enough (briefing, "Why 80/10/10").

The library's `DataCollatorForLanguageModeling(tokenizer, mlm_probability=0.15, mask_replace_prob=0.8, random_replace_prob=0.1)` implements the same rule; you now know what it does.

</details>

In [ ]:
# Checkpoint 4: a padded batch of 256 training texts.
workshop.checkpoint(4)
batch = mlm_tokenizer(train_texts[:256], padding=True, truncation=True, max_length=256,
                      return_special_tokens_mask=True, return_tensors="pt")
ids = batch["input_ids"]  # (256, T)
special = batch["special_tokens_mask"].bool()  # (256, T): [CLS], [SEP] and padding
torch.manual_seed(SEED)
corrupted, labels = mask_tokens(ids, special, mlm_tokenizer.mask_token_id, len(mlm_tokenizer))

sel = labels != -100  # the selected set S
assert corrupted.shape == ids.shape and labels.shape == ids.shape
assert not sel[special].any(), "a special or padding position was selected"
assert (labels[sel] == ids[sel]).all(), "labels must hold the original token at selected positions"
assert (corrupted[~sel] == ids[~sel]).all(), "positions outside S must not change"
share_selected = (sel.sum() / (~special).sum()).item()
share_mask = (corrupted[sel] == mlm_tokenizer.mask_token_id).float().mean().item()
share_unchanged = (corrupted[sel] == ids[sel]).float().mean().item()
assert 0.13 <= share_selected <= 0.17, f"selected share {share_selected:.3f}, expected about 0.15"
assert 0.77 <= share_mask <= 0.83, f"[MASK] share of S {share_mask:.3f}, expected about 0.80"
assert 0.08 <= share_unchanged <= 0.125, f"unchanged share {share_unchanged:.3f}, expected 0.10"
print("Checkpoint 4 passed")
print(f"{int((~special).sum()):,} real tokens, {int(sel.sum()):,} selected ({share_selected:.3f}); "
      f"of those: [MASK] {share_mask:.3f}, unchanged {share_unchanged:.3f}, "
      f"random {1 - share_mask - share_unchanged:.3f}")

**Run: the masked-LM loss.** @eq-mlm averages the negative log-probability of the original token over the selected positions only:

$$
\mathcal{L}_{\text{MLM}}(\theta) = -\frac{1}{|\mathcal{S}|} \sum_{t \in \mathcal{S}} \log P_\theta\bigl(w_t \mid \tilde{w}_1, \dots, \tilde{w}_T\bigr).
$$

The cell corrupts 16 validation abstracts with your `mask_tokens`, asks the pretrained model for the loss, and recomputes it by hand from the logits.

In [ ]:
_b = mlm_tokenizer(val_texts[:16], padding=True, truncation=True, max_length=256,
                   return_special_tokens_mask=True, return_tensors="pt")
torch.manual_seed(SEED)
_corrupted, _labels = mask_tokens(_b["input_ids"], _b["special_tokens_mask"].bool(),
                                  mlm_tokenizer.mask_token_id, len(mlm_tokenizer))
with torch.no_grad():
    out = mlm(input_ids=_corrupted.to(DEVICE), attention_mask=_b["attention_mask"].to(DEVICE),
              labels=_labels.to(DEVICE))
# @eq-mlm by hand: cross-entropy over the selected positions; -100 marks the rest
by_hand = F.cross_entropy(out.logits.flatten(0, 1).float(), _labels.to(DEVICE).flatten(),
                          ignore_index=-100)
assert math.isclose(out.loss.item(), by_hand.item(), rel_tol=1e-4)
print(f"Masked-LM loss on {int((_labels != -100).sum())} selected positions of 16 abstracts: "
      f"{out.loss.item():.2f} nats (by hand: {by_hand.item():.2f}); "
      f"a uniform guess would score log |V| = {math.log(len(mlm_tokenizer)):.2f}")

**Run: fill in the blank.** The pretrained encoder returns a distribution over the vocabulary at the masked position, not one word.

In [ ]:
for text in [f"We train a neural {mlm_tokenizer.mask_token} on a large corpus of text.",
             f"The robot uses a {mlm_tokenizer.mask_token} to grasp objects on the table.",
             f"The {mlm_tokenizer.mask_token} was trained on images of street scenes."]:
    inputs = mlm_tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = mlm(**inputs).logits  # (1, T, |V|)
    t = (inputs["input_ids"][0] == mlm_tokenizer.mask_token_id).nonzero().item()
    probs = logits[0, t].softmax(dim=-1)  # distribution over V at the masked position
    top = probs.topk(5)
    guesses = ", ".join(f"{mlm_tokenizer.convert_ids_to_tokens(int(i))} {p.item():.2f}"
                        for p, i in zip(top.values, top.indices, strict=True))
    print(f"{text}\n   -> {guesses}")

**Explain.** Read the loss against the uniform baseline: the encoder recovers hidden tokens far better than chance, using context from both sides. Notice what the encoder cannot do: it has no perplexity for a whole sentence, because @eq-mlm does not factor $P(w_1, \dots, w_T)$, and it does not generate text left to right. That is the trade made for representations in which every position has read the whole input, which is what classification wants.

# Part C · Fine-tuning for classification

## The baseline, recomputed

Lab 1's TF-IDF + logistic regression, with Lab 1's settings (its tokenizer, `min_df=2`, `C = 10` as chosen on val there), refitted here so the notebook runs on its own. It takes about a second.

In [ ]:
def lab1_tokenize(text):
    """Lab 1's word tokenizer."""
    return re.findall(r"[a-z0-9']+|[^a-z0-9'\s]", text.lower())


y_train, y_val, y_test = np.array(train_labels), np.array(val_labels), np.array(test_labels)
started = time.time()
_vectorizer = CountVectorizer(tokenizer=lab1_tokenize, lowercase=False, token_pattern=None,
                              min_df=2)
_tfidf = TfidfTransformer()
X_train = _tfidf.fit_transform(_vectorizer.fit_transform(train_texts))
X_test = _tfidf.transform(_vectorizer.transform(test_texts))
_baseline = LogisticRegression(C=10.0, max_iter=1000, random_state=SEED).fit(X_train, y_train)
tfidf_pred = _baseline.predict(X_test)
tfidf_result = {"accuracy": accuracy_score(y_test, tfidf_pred),
                "macro_f1": f1_score(y_test, tfidf_pred, average="macro")}

RECORDED_TFIDF_ACCURACY = 0.8838  # data/baselines.json, lab01.tfidf_logreg
assert abs(tfidf_result["accuracy"] - RECORDED_TFIDF_ACCURACY) < 5e-3, tfidf_result
print(f"TF-IDF + logistic regression: test accuracy {tfidf_result['accuracy']:.4f}, "
      f"macro-F1 {tfidf_result['macro_f1']:.4f} ({time.time() - started:.1f} s)")

## Exercise 5 · Fine-tune an encoder (12 minutes)

The model is the encoder with one new output layer on the output vector at the `[CLS]` position (briefing, @eq-head):

$$
z = W\, h_{\texttt{[CLS]}} + b, \qquad \hat{y} = \mathrm{softmax}(z), \qquad \mathcal{L} = -\log \hat{y}_y .
$$

It is Lab 1's logistic regression with new features. All parameters, encoder and head, are trained together with a small learning rate for a few epochs. `AutoModelForSequenceClassification` adds the head with random parameters and prints a report saying so: that report is expected here.

**How much does truncation cut?** Texts are truncated *inside the tokenizer call* to `MAX_LENGTH` tokens; the data is never cut. The next cell measures how many training texts are longer than that.

In [ ]:
MAX_LENGTH = 256
ft_tokenizer = AutoTokenizer.from_pretrained(FINETUNE_ENCODER)
# Token counts include [CLS] and [SEP].
_lengths = np.array([len(ids) for ids in ft_tokenizer(train_texts)["input_ids"]])
print(f"{FINETUNE_ENCODER} tokens per training text: median {np.median(_lengths):.0f}, "
      f"95th percentile {np.percentile(_lengths, 95):.0f}, maximum {_lengths.max()}")
print(f"{(_lengths > MAX_LENGTH).mean():.0%} of training texts are longer than {MAX_LENGTH} tokens "
      "and lose their end; the title and opening sentences, which carry most of the topic, "
      "are kept.")

**Predict.** The new head is random, so before any training the model's predicted distribution is close to uniform over $C = 4$ classes. What will the validation loss be before the first step? Write down a number.

**Task.** Complete the two functions that `Trainer` calls.

- `tokenize(batch)`: `batch["text"]` is a list of strings. Return `ft_tokenizer`'s output with truncation at `MAX_LENGTH` and **no padding**: the collator pads each batch to its own longest text later.
- `compute_metrics(eval_pred)`: `eval_pred` unpacks to `(logits, labels)`, NumPy arrays of shape `(N, C)` and `(N,)`. Return `{"accuracy": ..., "macro_f1": ...}`, computed with scikit-learn as in Lab 1.

In [ ]:
# TODO 5: complete both functions.
def tokenize(batch):
    """Tokenize batch["text"] with ft_tokenizer, truncated to MAX_LENGTH tokens, unpadded."""
    raise NotImplementedError("TODO 5")


def compute_metrics(eval_pred):
    """Return {"accuracy": float, "macro_f1": float} from (logits, labels)."""
    logits, labels = eval_pred  # NumPy arrays: (N, C) and (N,)
    ...

In [ ]:
#@title Solution 5 — try it yourself first { display-mode: "form" }
@workshop.solution(5)
def tokenize(batch):
    """Tokenize batch["text"] with ft_tokenizer, truncated to MAX_LENGTH tokens, unpadded."""
    return ft_tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)


@workshop.solution(5)
def compute_metrics(eval_pred):
    """Return {"accuracy": float, "macro_f1": float} from (logits, labels)."""
    logits, labels = eval_pred  # NumPy arrays: (N, C) and (N,)
    pred = np.argmax(logits, axis=-1)  # softmax is monotone, so the largest logit is the prediction
    return {"accuracy": accuracy_score(labels, pred),
            "macro_f1": f1_score(labels, pred, average="macro")}

<details>
<summary>Why this solution works</summary>

`truncation=True, max_length=MAX_LENGTH` keeps `[CLS]`, the first `MAX_LENGTH - 2` text tokens and `[SEP]`. Leaving padding to `DataCollatorWithPadding` means each batch is padded only to its own longest text, which saves computation on short abstracts; the attention mask then tells the model which positions are padding (Module 4's padding mask).

`compute_metrics` needs no softmax: the largest logit is also the largest probability. Accuracy and macro-F1 come from the same scikit-learn calls as Lab 1, so the numbers in the results table are comparable row by row.

</details>

In [ ]:
# Checkpoint 5a
workshop.checkpoint(5, label="5a")
_logits = np.array([[3.0, 0, 0, 0], [0, 2.0, 0, 0], [0, 1.0, 0, 0], [0, 0, 1.5, 0], [0, 0, 0, 4.0]])
_labels = np.array([0, 0, 1, 2, 3])  # predictions 0, 1, 1, 2, 3: one error
_m = compute_metrics((_logits, _labels))
assert math.isclose(_m["accuracy"], 0.8), _m
assert math.isclose(_m["macro_f1"], (2 / 3 + 2 / 3 + 1 + 1) / 4), _m  # F1 per class: 2/3, 2/3, 1, 1
_enc = tokenize({"text": [train_texts[0], "A short title"]})
assert len(_enc["input_ids"][0]) <= MAX_LENGTH, "truncate to MAX_LENGTH"
assert len(_enc["input_ids"][1]) < len(_enc["input_ids"][0]), "do not pad: lengths should differ"
assert _enc["input_ids"][1][0] == ft_tokenizer.cls_token_id, "keep the special tokens"
print("Checkpoint 5a passed")

**Run: fine-tuning.** The whole step is one function, `finetune(model_name, max_length, epochs, lr, seed, train_rows)`, so that Lab 11 can reuse it. It uses your `tokenize` and `compute_metrics`, records the validation loss before training, trains, and returns the raw validation and test logits.

| Runtime | Encoder | Training data | Epochs | Learning rate | Batch |
|---|---|---|---|---|---|
| GPU | DistilBERT | all 4,800 training texts | 2 | 5e-5 | 32, `fp16` |
| CPU | BERT-mini | the first `CPU_TRAIN_ROWS` training texts, to fit the time budget | 3 | 1e-4 | 32 |

Validation and test are never reduced. On a CPU runtime validation is evaluated only before and after training, to save time. On a T4 this cell should take a few minutes; it prints the time it took.

In [ ]:
def finetune(model_name, max_length, epochs, lr, seed, train_rows=None):
    """Fine-tune `model_name` for 4-way topic classification and return its val and test logits.

    Uses tokenize() and compute_metrics() from Exercise 5. train_rows=None uses all 4,800
    training texts; a number uses the first train_rows. Val and test are always complete.
    """
    global ft_tokenizer, MAX_LENGTH  # read by tokenize()
    ft_tokenizer, MAX_LENGTH = AutoTokenizer.from_pretrained(model_name), max_length
    transformers.set_seed(seed)
    n = len(train_texts) if train_rows is None else train_rows
    train_ds = Dataset.from_dict({"text": train_texts[:n], "label": train_labels[:n]})
    val_ds = Dataset.from_dict({"text": val_texts, "label": val_labels})
    test_ds = Dataset.from_dict({"text": test_texts, "label": test_labels})
    train_ds, val_ds, test_ds = (d.map(tokenize, batched=True) for d in (train_ds, val_ds, test_ds))

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)  # new head
    args = TrainingArguments(
        output_dir="lab06", learning_rate=lr, num_train_epochs=epochs,
        per_device_train_batch_size=32, per_device_eval_batch_size=64,
        weight_decay=0.01, logging_strategy="epoch",
        eval_strategy="epoch" if DEVICE == "cuda" else "no",  # CPU: skip per-epoch evaluation
        save_strategy="no", report_to="none", seed=seed, fp16=(DEVICE == "cuda"),
        dataloader_pin_memory=(DEVICE == "cuda"),
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=DataCollatorWithPadding(tokenizer=ft_tokenizer),  # pads to batch's longest
        compute_metrics=compute_metrics,
    )
    start_val_loss = trainer.evaluate()["eval_loss"]  # random head, before any step
    started = time.time()
    trainer.train()
    train_seconds = time.time() - started
    return {
        "val_logits": np.asarray(trainer.predict(val_ds).predictions, dtype=np.float32),  # (600, 4)
        "val_labels": np.array(val_labels, dtype=np.int64),
        "test_logits": np.asarray(trainer.predict(test_ds).predictions, dtype=np.float32),
        "test_labels": np.array(test_labels, dtype=np.int64),
        "trainer": trainer, "start_val_loss": start_val_loss, "train_seconds": train_seconds,
    }


CPU_TRAIN_ROWS = 800  # CPU runtime only: 25 batches of 32 per epoch
if DEVICE == "cuda":
    RUN = dict(model_name=FINETUNE_ENCODER, max_length=256, epochs=2, lr=5e-5, seed=SEED,
               train_rows=None)
else:
    RUN = dict(model_name=FINETUNE_ENCODER, max_length=256, epochs=3, lr=1e-4, seed=SEED,
               train_rows=CPU_TRAIN_ROWS)
print("Fine-tuning:", RUN)
result = finetune(**RUN)
trainer = result["trainer"]
print(f"Training took {result['train_seconds']:.0f} s")

In [ ]:
# Checkpoint 5b: the starting loss, and @eq-head recomputed by hand.
workshop.checkpoint(5, label="5b")
print(f"Validation loss before training: {result['start_val_loss']:.3f}  "
      f"(log 4 = {math.log(4):.3f})")
assert abs(result["start_val_loss"] - math.log(4)) < 0.25, "a random head should start near log C"

model = trainer.model.eval()
_b = DataCollatorWithPadding(tokenizer=ft_tokenizer)(
    [ft_tokenizer(t, truncation=True, max_length=MAX_LENGTH) for t in val_texts[:8]]
).to(model.device)
with torch.no_grad():
    library_logits = model(**_b).logits.float()  # (8, 4)
    h = model.base_model(**_b).last_hidden_state  # (8, T, d): one h_t per token
    h_cls = h[:, 0]  # (8, d): the [CLS] position
    if model.config.model_type == "distilbert":  # [CLS] -> Linear(d, d) -> ReLU -> Linear(d, C)
        hidden = torch.relu(model.pre_classifier(h_cls))
    else:  # BERT: [CLS] -> pooler Linear(d, d) -> tanh -> Linear(d, C)
        hidden = torch.tanh(model.base_model.pooler.dense(h_cls))
    z = (hidden @ model.classifier.weight.T + model.classifier.bias).float()  # z = W h + b
assert torch.allclose(z, library_logits, atol=1e-3), (z - library_logits).abs().max()
print("Checkpoint 5b passed: the model's logits are W h + b on the [CLS] vector")

## The results table

One table, the same 1,600 test papers, the same two metrics. The Lab 2 row (averaged skip-gram embeddings and a feed-forward network, `lab02.sgns_avg_ffn`) is read from the workshop's recorded baselines (`data/baselines.json`), not retrained here.

In [ ]:
def recorded_baseline(baseline_id):
    """The entry of data/baselines.json with this id, or None if it is not recorded."""
    candidates = [Path(os.environ.get("NLP_LLMS_DATA", "data")) / "baselines.json",
                  Path("../data/baselines.json")]
    for path in candidates:
        if path.exists():
            doc = json.loads(path.read_text(encoding="utf-8"))
            break
    else:
        try:
            url = ("https://raw.githubusercontent.com/project-delphi/nlp-llms/main/"
                   "data/baselines.json")
            with urllib.request.urlopen(url, timeout=10) as response:
                doc = json.loads(response.read())
        except (OSError, ValueError):
            return None
    return next((b for b in doc["baselines"] if b["id"] == baseline_id), None)


test_pred = result["test_logits"].argmax(axis=1)
encoder_result = {"accuracy": accuracy_score(y_test, test_pred),
                  "macro_f1": f1_score(y_test, test_pred, average="macro")}
val_accuracy = accuracy_score(y_val, result["val_logits"].argmax(axis=1))
rows = "all 4,800" if RUN["train_rows"] is None else f"first {RUN['train_rows']:,} of 4,800"
encoder_name = (f"{RUN['model_name']}, max_length {RUN['max_length']}, "
                f"{RUN['epochs']} epochs, seed {RUN['seed']}, train rows: {rows}")
if OFFLINE_TINY:
    encoder_name = "TINY RANDOM STAND-IN, NOT A RESULT: " + encoder_name

lab2 = recorded_baseline("lab02.sgns_avg_ffn")  # Lab 2 core path, not its GloVe stretch
N_TEST = len(y_test)
print(f"arXiv Topics v1, test split ({N_TEST:,} papers)\n")
print(f"{'model':<44} {'accuracy':>9} {'macro-F1':>9} {'errors':>7}")
print(f"{'TF-IDF + logistic regression (Lab 1)':<44} {tfidf_result['accuracy']:>9.4f} "
      f"{tfidf_result['macro_f1']:>9.4f} {int((tfidf_pred != y_test).sum()):>7}")
if lab2:
    acc, mf1 = lab2["metrics"]["accuracy"], lab2["metrics"]["macro_f1"]
    print(f"{'Averaged embeddings + network (Lab 2)':<44} {acc:>9.4f} {mf1:>9.4f} "
          f"{round((1 - acc) * N_TEST):>7}")
else:
    print(f"{'Averaged embeddings + network (Lab 2)':<44} {'not recorded yet':>27}")
print(f"{'Fine-tuned encoder (this lab)':<44} {encoder_result['accuracy']:>9.4f} "
      f"{encoder_result['macro_f1']:>9.4f} {int((test_pred != y_test).sum()):>7}")
print(f"  {encoder_name}")
print(f"  val accuracy {val_accuracy:.4f} · training {result['train_seconds']:.0f} s on {DEVICE}")
diff = encoder_result["accuracy"] - tfidf_result["accuracy"]
print(f"\nEncoder minus TF-IDF: {100 * diff:+.1f} accuracy points "
      f"({round(diff * N_TEST):+d} correct test papers)")

In [ ]:
# Checkpoint 5c: training worked. The floors catch a broken run; they do not promise a gain over
# TF-IDF, which this lab does not claim. The lab brief quotes single-seed reference runs (full
# training split, not a T4) of 0.899 for DistilBERT and 0.886 for BERT-mini; neither has been
# reproduced in the repository's build, so treat both as unverified until measured on Colab.
workshop.checkpoint(5, label="5c")
ACCURACY_FLOOR = {"cuda": 0.85, "cpu": 0.60}  # provisional until 3 seeds are measured on Colab
if not OFFLINE_TINY:
    assert encoder_result["accuracy"] >= ACCURACY_FLOOR[DEVICE], encoder_result
    print(f"Checkpoint 5c passed: test accuracy {encoder_result['accuracy']:.4f} "
          f">= {ACCURACY_FLOOR[DEVICE]} on {DEVICE}")
else:
    print("Checkpoint 5c skipped: offline test mode has no pretrained model")

**Explain.** Write two sentences in the cell below (double-click to edit):

1. By how many test papers does the encoder differ from TF-IDF? Is that more than you would expect from changing the random seed? (Half a point is eight papers.)
2. What does the encoder buy, and at what price? Compare the training time with the one second TF-IDF needs.

*Your answer:*

Things to notice. Topic classification depends mostly on which words appear, and a bag of words captures that well, so a fine-tuned encoder typically matches TF-IDF or beats it by a small margin. A difference of a few papers is within seed-to-seed variation; one run cannot show a gain. The stronger case for pretraining is elsewhere: no feature engineering, fewer labels needed, and tasks where word order decides the answer. On a CPU runtime the encoder is smaller and saw a fraction of the training data, so its row is not a fair comparison with TF-IDF on all 4,800 texts.

## Hand-off to Lab 11

Lab 11 asks whether this classifier's probabilities are calibrated: does a predicted 0.9 mean right nine times in ten? It needs the **raw logits** (temperature scaling divides them) on validation, to fit, and on test, to evaluate. A Colab runtime does not persist, so the next cell writes them to `lab06_logits.npz`, with the settings that produced them, and saves the fine-tuned model to `lab06-classifier/` for anyone who keeps this runtime open. Lab 11 loads a copy recorded by the workshop by default, so you do not have to carry the file over.

In [ ]:
meta = {
    "model": RUN["model_name"], "max_length": RUN["max_length"], "epochs": RUN["epochs"],
    "lr": RUN["lr"], "seed": RUN["seed"], "train_rows": RUN["train_rows"] or len(train_texts),
    "device": DEVICE, "offline_tiny_stand_in": OFFLINE_TINY,
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                 "tokenizers": tokenizers.__version__, "datasets": datasets.__version__},
    "test_accuracy": round(float(encoder_result["accuracy"]), 4),
}
np.savez("lab06_logits.npz", val_logits=result["val_logits"], val_labels=result["val_labels"],
         test_logits=result["test_logits"], test_labels=result["test_labels"],
         meta=json.dumps(meta))
trainer.save_model("lab06-classifier")

with np.load("lab06_logits.npz") as saved:
    assert saved["val_logits"].shape == (600, 4) and saved["val_logits"].dtype == np.float32
    assert saved["test_logits"].shape == (1600, 4) and saved["test_labels"].dtype == np.int64
    assert (saved["test_labels"] == y_test).all() and (saved["val_labels"] == y_val).all()
print(f"Wrote lab06_logits.npz ({Path('lab06_logits.npz').stat().st_size / 1e3:.0f} kB) "
      "and lab06-classifier/")
print(json.dumps(meta, indent=1))
print(f"\nThis is the end of the core path. Run time so far: {time.time() - LAB_START:.0f} s")

## Stretch (optional) · Attention and hidden states of a pretrained encoder

For those who finish early. Nothing later in the workshop depends on this section.

`AutoModel` loads the encoder body without a head. Called with `output_attentions=True` and `output_hidden_states=True`, it returns the attention weights of Module 5, one tensor of shape `(1, heads, T, T)` per layer, and the vectors $h_t$ after every layer, starting with the embeddings. The default attention implementation computes attention in a fused kernel and returns no weights, so the model is loaded with `attn_implementation="eager"`.

A useful summary of one head is the **entropy** of its attention rows: for row $t$ with weights $a_{t,1}, \dots, a_{t,T}$, $H_t = -\sum_s a_{t,s} \log a_{t,s}$. A head that spreads attention evenly over $T$ tokens has entropy $\log T$; a head that looks at one token has entropy 0.

**Predict.** Will heads in the first layer or in the last layer attend more broadly (higher entropy)?

**Task.** Complete `attention_entropy(attn)` for one head's weights of shape `(T, T)`: the mean over rows of the row entropy. Treat $0 \log 0$ as 0.

In [ ]:
encoder = AutoModel.from_pretrained(MLM_ENCODER, attn_implementation="eager").to(DEVICE).eval()
text = val_texts[0].split("\n")[0]  # the title of one validation paper
inputs = mlm_tokenizer(text, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    out = encoder(**inputs, output_attentions=True, output_hidden_states=True)
tokens = mlm_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(text)
print(f"{len(out.attentions)} attention tensors of shape {tuple(out.attentions[0].shape)}; "
      f"{len(out.hidden_states)} hidden-state tensors of shape {tuple(out.hidden_states[0].shape)}")

In [ ]:
# TODO 6 (stretch): complete this function.
def attention_entropy(attn):
    """attn: (T, T), each row sums to 1. Return the mean row entropy in nats, as a float."""
    raise NotImplementedError("TODO 6")

In [ ]:
#@title Solution 6 — try it yourself first { display-mode: "form" }
@workshop.solution(6)
def attention_entropy(attn):
    """attn: (T, T), each row sums to 1. Return the mean row entropy in nats, as a float."""
    attn = attn.float()
    plogp = torch.where(attn > 0, attn * attn.clamp_min(1e-30).log(), torch.zeros_like(attn))
    return (-plogp.sum(dim=-1)).mean().item()  # entropy of each row, then the mean over rows

<details>
<summary>Why this solution works</summary>

Each row of a head's weights is a probability distribution over the positions it reads from: the softmax of Module 5's scaled scores. Its entropy measures how spread out that distribution is. `torch.where` sets the $0 \log 0$ terms to 0 instead of `nan`, and `clamp_min` keeps the logarithm finite where the weight is 0 anyway.

</details>

In [ ]:
# Checkpoint 6 (stretch)
workshop.checkpoint(6)
for layer, attn in enumerate(out.attentions):
    assert torch.allclose(attn.sum(-1), torch.ones_like(attn.sum(-1)), atol=1e-5), layer
_T = 7
assert math.isclose(attention_entropy(torch.full((_T, _T), 1 / _T)), math.log(_T), rel_tol=1e-6)
assert attention_entropy(torch.eye(_T)) == 0.0, "a head that attends to one token has entropy 0"
print("Checkpoint 6 passed")

entropy = np.array([[attention_entropy(attn[0, h]) for h in range(attn.shape[1])]
                    for attn in out.attentions])  # (layers, heads)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
im = axes[0].imshow(entropy, cmap="Blues", aspect="auto")
axes[0].set_xlabel("head")
axes[0].set_ylabel("layer")
axes[0].set_title(f"Mean attention entropy (max log T = {math.log(len(tokens)):.2f})")
fig.colorbar(im, ax=axes[0])
lowest = np.unravel_index(entropy.argmin(), entropy.shape)
highest = np.unravel_index(entropy.argmax(), entropy.shape)
for ax, (layer, head) in zip(axes[1:], [lowest, highest], strict=True):
    ax.imshow(out.attentions[layer][0, head].float().cpu(), cmap="Blues")
    ax.set_xticks(range(len(tokens)), tokens, rotation=90, fontsize=7)
    ax.set_yticks(range(len(tokens)), tokens, fontsize=7)
    ax.set_title(f"layer {layer + 1}, head {head + 1}: entropy {entropy[layer, head]:.2f}")
plt.tight_layout()
plt.show()
print("Mean entropy per layer:", np.round(entropy.mean(axis=1), 2).tolist())

**Run: one word, two contexts.** A static embedding gives *bank* one vector. A contextual encoder gives it a different $h_t$ in each sentence. The cell compares the vectors of the same word in two sentences, layer by layer, with cosine similarity.

In [ ]:
def word_vectors(sentence, word):
    """The hidden states of `word` in `sentence`, one (d,) vector per layer (pieces averaged)."""
    start = re.search(rf"\b{word}\b", sentence).start()  # character span of the word
    end = start + len(word)
    enc = mlm_tokenizer(sentence, return_tensors="pt", return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")[0].tolist()  # (T, 2) character span of each token
    positions = [i for i, (s, e) in enumerate(offsets) if e > s and s >= start and e <= end]
    with torch.no_grad():
        states = encoder(**enc.to(DEVICE), output_hidden_states=True).hidden_states
    return [s[0, positions].mean(0) for s in states]  # layers + 1 vectors of shape (d,)


# "bank" is the fifth token in all three sentences, so it has the same position embedding:
# any difference between the vectors comes from the context.
a = word_vectors("She sat on the bank of the river and watched the water.", "bank")
b = word_vectors("She went to the bank to ask for a loan.", "bank")
c = word_vectors("They sat on the bank of the stream after lunch.", "bank")
print("cosine similarity of 'bank' across contexts, by layer (0 = embeddings)")
print("layer   river vs loan   river vs stream")
for layer, (va, vb, vc) in enumerate(zip(a, b, c, strict=True)):
    print(f"{layer:>5}   {F.cosine_similarity(va, vb, dim=0).item():>13.3f}   "
          f"{F.cosine_similarity(va, vc, dim=0).item():>15.3f}")

**Explain.** In one sentence: which layer separates the river sense of *bank* from the money sense, and does the river-versus-stream pair stay closer? At layer 0 the three vectors should be identical (cosine 1.000): same word, same position, and no context yet. Every later layer mixes in the rest of the sentence. Carry over Module 4's caution: attention weights show where a head read, not why the model decided anything.

In [ ]:
# Generated by scripts/gen_notebooks.py: do not edit this cell.
# Which checkpoints passed, and on whose code; then a run record for the instructor.
workshop.summary()
workshop.run_record()

<!-- GENERATED by scripts/gen_notebooks.py. Do not edit this cell. -->
---
**Next:** [7 · Fine-tuning and LoRA](https://colab.research.google.com/github/project-delphi/nlp-llms/blob/main/notebooks/07-finetuning-lora.ipynb)

[Workshop site](https://project-delphi.github.io/nlp-llms) · [All notebooks](https://project-delphi.github.io/nlp-llms/notebooks.html) · [Source](https://github.com/project-delphi/nlp-llms)

Genial Labs · text CC BY 4.0, code MIT